# Ensemble methods. Exercises


In this section we have only two exercise:

1. Find the best three classifier in the stacking method using the classifiers from scikit-learn package.

2. Build arcing arc-x4 method. 

In [1]:
%store -r data_set
%store -r labels
%store -r test_data_set
%store -r test_labels
%store -r unique_labels

## Exercise 1: Find the best three classifier in the stacking method

Please use the following classifiers:

* Linear regression,
* Nearest Neighbors,
* Linear SVM,
* Decision Tree,
* Naive Bayes,
* QDA.

In [2]:
import numpy as np
from sklearn.metrics import accuracy_score

from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.discriminant_analysis import QuadraticDiscriminantAnalysis

In [6]:
def build_classifiers():
    clf1 = KNeighborsClassifier()
    clf1.fit(data_set, labels)

    clf2 = GaussianNB()
    clf2.fit(data_set, labels)

    clf3 = DecisionTreeClassifier(random_state=42)
    clf3.fit(data_set, labels)

    return [clf1, clf2, clf3]

In [7]:
def build_stacked_classifier(classifiers):
    output = []
    for classifier in classifiers:
        output.append(classifier.predict(data_set))
    output = np.array(output).reshape((130,3))
    
    # stacked classifier part:
    stacked_classifier = DecisionTreeClassifier(random_state=42)
    stacked_classifier.fit(output.reshape((130,3)), labels.reshape((130,)))
    test_set = []
    for classifier in classifiers:
        test_set.append(classifier.predict(test_data_set))
    test_set = np.array(test_set).reshape((len(test_set[0]),3))
    predicted = stacked_classifier.predict(test_set)
    return predicted

In [8]:
classifiers = build_classifiers()
predicted = build_stacked_classifier(classifiers)
accuracy = accuracy_score(test_labels, predicted)
print(accuracy)

0.85


## Exercise 2: 

Use the boosting method and change the code to fullfilt the following requirements:

* the weights should be calculated as:
$w_{n}^{(t+1)}=\frac{1+ I(y_{n}\neq h_{t}(x_{n})}{\sum_{i=1}^{N}1+I(y_{n}\neq h_{t}(x_{n})}$,
* the prediction is done with a voting method.

In [9]:
import numpy as np
from sklearn.tree import DecisionTreeClassifier

# prepare data set

def generate_data(sample_number, feature_number, label_number):
    data_set = np.random.random_sample((sample_number, feature_number))
    labels = np.random.choice(label_number, sample_number)
    return data_set, labels

labels = 2
dimension = 2
test_set_size = 1000
train_set_size = 5000
train_set, train_labels = generate_data(train_set_size, dimension, labels)
test_set, test_labels = generate_data(test_set_size, dimension, labels)

# init weights
number_of_iterations = 10
weights = np.ones((test_set_size,)) / test_set_size


def train_model(classifier, weights):
    return classifier.fit(X=test_set, y=test_labels, sample_weight=weights)

def calculate_error(model):
    predicted = model.predict(test_set)
    I=calculate_accuracy_vector(predicted, test_labels)
    Z=np.sum(I)
    return (1+Z)/1.0

Fill the two functions below:

In [10]:
def set_new_weights(model):
    predicted = model.predict(test_set)

    indicator = (predicted != test_labels).astype(int)

    new_weights = (1 + indicator) / np.sum(1 + indicator)

    return new_weights

Train the classifier with the code below:

In [11]:
classifier = DecisionTreeClassifier(max_depth=1, random_state=1)
classifier.fit(X=train_set, y=train_labels)
alphas = []
classifiers = []
for iteration in range(number_of_iterations):
    model = train_model(classifier, weights)
    weights = set_new_weights(model)
    classifiers.append(model)

print(weights)


validate_x, validate_label = generate_data(1, dimension, labels)

[0.00066269 0.00132538 0.00066269 0.00132538 0.00132538 0.00066269
 0.00132538 0.00066269 0.00066269 0.00132538 0.00066269 0.00132538
 0.00132538 0.00132538 0.00066269 0.00066269 0.00132538 0.00132538
 0.00132538 0.00132538 0.00132538 0.00066269 0.00132538 0.00066269
 0.00132538 0.00132538 0.00132538 0.00132538 0.00132538 0.00066269
 0.00066269 0.00066269 0.00132538 0.00066269 0.00066269 0.00066269
 0.00066269 0.00132538 0.00132538 0.00132538 0.00132538 0.00066269
 0.00066269 0.00066269 0.00132538 0.00066269 0.00132538 0.00066269
 0.00066269 0.00132538 0.00066269 0.00132538 0.00066269 0.00132538
 0.00132538 0.00066269 0.00066269 0.00066269 0.00132538 0.00066269
 0.00132538 0.00132538 0.00066269 0.00132538 0.00132538 0.00132538
 0.00066269 0.00132538 0.00066269 0.00066269 0.00132538 0.00132538
 0.00066269 0.00132538 0.00066269 0.00132538 0.00132538 0.00132538
 0.00066269 0.00066269 0.00066269 0.00066269 0.00066269 0.00066269
 0.00132538 0.00132538 0.00066269 0.00066269 0.00132538 0.0006

Set the validation data set:

In [12]:
validate_x, validate_label = generate_data(1, dimension, labels)

Fill the prediction code:

In [13]:
def get_prediction(x):
    predictions = np.array([clf.predict(x) for clf in classifiers])

    majority_votes = [np.bincount(predictions[:, i]).argmax() for i in range(predictions.shape[1])]

    return np.array(majority_votes)

Test it:

In [14]:
prediction = get_prediction(validate_x)[0]

print(prediction)

1
